# 001: Dataset Pipeline Engineering — Data loading, splitting, stratified sampling, and augmentations

## 1. PIPELINE CORE
- In production ML, datasets must be processed in pipelines to prevent data leakage and ensure reproducibility.
- **Train / Val / Test Splits**:
  - Train: Used to compute gradients and update weights.
  - Validation: Used for hyperparameter tuning and early stopping checks.
  - Test: Final evaluation to measure generalizability. Standard split ratio is 70/15/15 or 80/10/10.
- **Stratified Sampling**:
  - Ensures that each split contains approximately the same percentage of samples of each target class. Critical for class-imbalanced datasets to prevent evaluation bias.

## 2. DATA AUGMENTATION
- Artificially increases training dataset size and variance by applying random transformations (e.g. flipping, rotation, adding gaussian noise).
- Acts as a regularizer, forcing the network to learn invariant features rather than spatial coordinate patterns.
---


In [ ]:
import numpy as np

class DatasetPipeline:
    """Enterprise-grade dataset pipeline from scratch."""
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def train_test_split(self, val_ratio=0.15, test_ratio=0.15, stratify=True):
        """Splits data into train, validation, and test sets with stratification option."""
        n_samples = self.X.shape[0]
        indices = np.arange(n_samples)
        
        if stratify:
            # Group indices by class
            unique_classes = np.unique(self.y)
            train_idx, val_idx, test_idx = [], [], []
            
            for c in unique_classes:
                class_indices = indices[self.y == c]
                np.random.shuffle(class_indices)
                
                n_class = len(class_indices)
                n_val = int(n_class * val_ratio)
                n_test = int(n_class * test_ratio)
                
                test_idx.extend(class_indices[:n_test])
                val_idx.extend(class_indices[n_test:n_test+n_val])
                train_idx.extend(class_indices[n_test+n_val:])
        else:
            np.random.shuffle(indices)
            n_val = int(n_samples * val_ratio)
            n_test = int(n_samples * test_ratio)
            
            test_idx = indices[:n_test]
            val_idx = indices[n_test:n_test+n_val]
            train_idx = indices[n_test+n_val:]
            
        return (self.X[train_idx], self.y[train_idx]), \
               (self.X[val_idx], self.y[val_idx]), \
               (self.X[test_idx], self.y[test_idx])



In [ ]:
# ── Data Augmentation Module ──

class AudioDataAugmenter:
    """Applies common signal augmentations to 1D data."""
    @staticmethod
    def add_white_noise(data, noise_level=0.05):
        noise = np.random.randn(*data.shape) * noise_level
        return data + noise

    @staticmethod
    def time_shift(data, shift_max=10):
        shift = np.random.randint(-shift_max, shift_max)
        return np.roll(data, shift)



In [ ]:
# Testing the pipeline and augmentations
print("--- Stratified Split Test ---")
np.random.seed(42)
# Create synthetic imbalanced dataset: 100 samples, 20% class 1, 80% class 0
X_synth = np.random.randn(100, 5)
y_synth = np.array([1]*20 + [0]*80)

pipeline = DatasetPipeline(X_synth, y_synth)
(X_tr, y_tr), (X_val, y_val), (X_te, y_te) = pipeline.train_test_split(val_ratio=0.15, test_ratio=0.15)

print(f"Train size: {len(y_tr)} | Class 1 proportion: {y_tr.mean():.2%}")
print(f"Val size:   {len(y_val)} | Class 1 proportion: {y_val.mean():.2%}")
print(f"Test size:  {len(y_te)} | Class 1 proportion: {y_te.mean():.2%}")

print("\n--- Audio Augmentation demo ---")
signal = np.sin(np.linspace(0, 2*np.pi, 20))
noisy = AudioDataAugmenter.add_white_noise(signal, noise_level=0.1)
shifted = AudioDataAugmenter.time_shift(signal, shift_max=3)

print(f"Original signal (first 5): {signal[:5].round(3)}")
print(f"Noisy signal (first 5):    {noisy[:5].round(3)}")
print(f"Shifted signal (first 5):  {shifted[:5].round(3)}")
